<a href="https://colab.research.google.com/github/sayanasurendran04/Revolutionizing-health-prediction-with-genomics/blob/main/Revolutionizing_health_prediction_with_genomics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ====================== 1. Synthetic Data Generation ======================
print("Generating synthetic genomic + lifestyle dataset...")
np.random.seed(42)
n_samples = 5000

X = np.zeros((n_samples, 25))
X[:, :20] = np.random.randint(0, 3, size=(n_samples, 20))          # 20 SNPs
X[:, 20] = np.random.normal(45, 15, n_samples).clip(18, 80)       # Age
X[:, 21] = np.random.normal(27, 5, n_samples).clip(15, 45)        # BMI
X[:, 22] = np.random.randint(0, 2, n_samples)                     # Smoking
X[:, 23] = np.random.randint(0, 4, n_samples)                     # Activity level
X[:, 24] = np.random.randint(0, 2, n_samples)                     # Family History

risk_score = (X[:, :20].sum(axis=1) * 0.15 +
              X[:, 20] * 0.02 +
              X[:, 21] * 0.08 +
              X[:, 22] * 1.5 +
              -X[:, 23] * 0.4 +
              X[:, 24] * 1.2)

y = (risk_score > risk_score.mean()).astype(int)

print(f"Dataset shape: {X.shape} | Positive cases: {y.sum()}/{n_samples}")

# ====================== 2. Preprocessing ======================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# Reshape for CNN (samples, features, 1 channel)
X_train_cnn = X_train.reshape(-1, 25, 1)
X_test_cnn  = X_test.reshape(-1, 25, 1)

# ====================== 3. Original Dense Model ======================
print("\n=== Training Dense Model ===")
model_dense = keras.Sequential([
    keras.layers.Input(shape=(25,)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(1, activation='sigmoid')
])

model_dense.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

model_dense.summary()

history_dense = model_dense.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5)
    ],
    verbose=1
)

# ====================== 4. CNN Model (1D Convolutional Neural Network) ======================
print("\n=== Training 1D CNN Model ===")
model_cnn = keras.Sequential([
    keras.layers.Input(shape=(25, 1)),

    # Convolutional layers - good for capturing local patterns in SNP sequences
    keras.layers.Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling1D(pool_size=2),

    keras.layers.Conv1D(filters=128, kernel_size=3, activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling1D(pool_size=2),

    keras.layers.Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
    keras.layers.BatchNormalization(),

    keras.layers.GlobalAveragePooling1D(),   # Reduces to fixed size vector

    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

model_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

model_cnn.summary()

history_cnn = model_cnn.fit(
    X_train_cnn, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5)
    ],
    verbose=1
)

# ====================== 5. Evaluation ======================
print("\n=== Evaluation Results ===")
dense_loss, dense_acc, dense_auc = model_dense.evaluate(X_test, y_test, verbose=0)
cnn_loss, cnn_acc, cnn_auc = model_cnn.evaluate(X_test_cnn, y_test, verbose=0)

print(f"Dense Model  →  Accuracy: {dense_acc:.4f} | AUC: {dense_auc:.4f}")
print(f"CNN Model    →  Accuracy: {cnn_acc:.4f} | AUC: {cnn_auc:.4f}")

# ====================== 6. Prediction Function (works for both models) ======================
def predict_risk(model, new_patient_data, is_cnn=False):
    """new_patient_data: array of shape (1, 25)"""
    scaled = scaler.transform(new_patient_data)
    if is_cnn:
        input_data = scaled.reshape(-1, 25, 1)
    else:
        input_data = scaled
    prob = model.predict(input_data, verbose=0)[0][0]
    risk = "HIGH" if prob > 0.5 else "LOW"
    return f"Predicted Disease Risk: {risk} ({prob:.1%} probability)"

# Example usage
sample_patient = np.array([[1,2,0,1]*5 + [55, 32, 1, 1, 1]])

print("\nDense Model Prediction:")
print(predict_risk(model_dense, sample_patient, is_cnn=False))

print("\nCNN Model Prediction:")
print(predict_risk(model_cnn, sample_patient, is_cnn=True))

Generating synthetic genomic + lifestyle dataset...
Dataset shape: (5000, 25) | Positive cases: 2528/5000

=== Training Dense Model ===


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         3,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,593 (57.00 KB)

 Trainable params: 14,145 (55.25 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 25, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 25, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 12, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 12, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 12, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 6, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 6, 64)          │        24,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 6, 64)          │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 56,897 (222.25 KB)

 Trainable params: 56,385 (220.25 KB)

 Non-trainable params: 512 (2.00 KB)

Epoch 1/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.7056 - auc: 0.7786 - loss: 0.5672 - val_accuracy: 0.7850 - val_auc: 0.8621 - val_loss: 0.6472 - learning_rate: 0.0010
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.8669 - auc: 0.9424 - loss: 0.3179 - val_accuracy: 0.7788 - val_auc: 0.8831 - val_loss: 0.5665 - learning_rate: 0.0010
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9222 - auc: 0.9803 - loss: 0.1838 - val_accuracy: 0.4925 - val_auc: 0.8671 - val_loss: 1.0586 - learning_rate: 0.0010
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9491 - auc: 0.9900 - loss: 0.1294 - val_accuracy: 0.5600 - val_auc: 0.8705 - val_loss: 1.1456 - learning_rate: 0.0010
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.9631 - auc: 0.9936 - loss: 0.1024 - val_accuracy: 0.6800 - val_auc: 0.9002 - val_loss: 0.8424 - learning_rate: 0.0010
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.9725 - auc: 0.9967 -

In [ ]:
!pip install cyvcf2 pandas numpy scikit-learn

from cyvcf2 import VCF
import pandas as pd
import numpy as np

def vcf_to_feature_matrix(vcf_path, sample_ids=None, max_variants=10000):
    vcf = VCF(vcf_path)

    genotypes = []
    variant_ids = []

    for i, variant in enumerate(vcf):
        if i >= max_variants:
            break
        # Get genotypes as numeric (0=ref/ref, 1=Het, 2=HomAlt, 3=Unknown)
        gt = variant.gt_types  # 0=HomRef, 1=Het, 2=HomAlt, 3=Unknown
        genotypes.append(gt)
        variant_ids.append(f"{variant.CHROM}:{variant.POS}")

    df = pd.DataFrame(np.array(genotypes).T, columns=variant_ids, index=vcf.samples)
    df = df.replace(3, np.nan).fillna(0)  # Handle missing

    return df

# Usage
genomic_df = vcf_to_feature_matrix("patient_data.vcf.gz", max_variants=5000)
print(genomic_df.shape)  # (n_patients, n_snps)

(2, 4)


In [ ]:
# Install bgzip and tabix
!apt-get update
!apt-get install -y samtools tabix bcftools

# Create a dummy VCF.gz file for demonstration purposes
# In a real scenario, you would replace this with your actual VCF file

vcf_content = """##fileformat=VCFv4.2
##fileDate=20230101
##source=dummy_generator
##contig=<ID=chr1,length=100000>
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\tSAMPLE_1\tSAMPLE_2
chr1\t100\t.\tA\tT\t.\tPASS\t.\tGT\t0/0\t0/1
chr1\t200\t.\tC\tG\t.\tPASS\t.\tGT\t1/1\t0/0
chr1\t300\t.\tG\tA\t.\tPASS\t.\tGT\t0/1\t1/1
chr1\t400\t.\tT\tC\t.\tPASS\t.\tGT\t0/0\t1/0
"""

with open("patient_data.vcf", "w") as f:
    f.write(vcf_content)

# Compress the VCF file
import subprocess
subprocess.run(["bgzip", "-f", "patient_data.vcf"])
subprocess.run(["tabix", "-p", "vcf", "patient_data.vcf.gz"])

print("Dummy patient_data.vcf.gz created.")

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 https://cli.github.com/packages stable InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,219 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,622 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,977 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,310 kB]
Get:14 https://r2u.